In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy

import itertools
from tqdm import tqdm

In [ ]:
rng = np.random.default_rng()

In [ ]:
N = 64
ms = range(1,40*N + 1,N//8)
num_trials = 30

test_m = 1000

lambda_fns = (
                lambda X : 0.01  * np.linalg.norm(X, ord='fro')**2 / (X.shape[1]),
                lambda X : 1e-3  * np.linalg.norm(X, ord=2)**2,
                lambda X : 0.01  * np.linalg.norm(X, ord='fro')**2 / (X.shape[0]**(1/3) * min(X.shape)**(2/3)),
                lambda X : 0.1  * np.linalg.norm(X, ord='fro')**2 / (X.shape[0]**(1/3) * min(X.shape)**(2/3)),
                lambda X : 1    * np.linalg.norm(X, ord='fro')**2 / (X.shape[0]**(1/3) * min(X.shape)**(2/3)),
                lambda X : 10   * np.linalg.norm(X, ord='fro')**2 / (X.shape[0]**(1/3) * min(X.shape)**(2/3)),
                lambda X : 100  * np.linalg.norm(X, ord='fro')**2 / (X.shape[0]**(1/3) * min(X.shape)**(2/3)),
)

labels = (
    '$\\lambda_1$',
    '$\\lambda_2$',
    '$\\beta = 0.01$',
    '$\\beta = 0.1$',
    '$\\beta = 1$',
    '$\\beta = 10$',
    '$\\beta = 100$',
)


def generate_nonuniform_hadamard(m, N):
    p = 1/np.arange(1,N+1)
    return scipy.linalg.hadamard(N)[rng.choice(N,size=(m,),p=p/np.sum(p)),:]

d_in = 16
W = rng.standard_normal(size=(d_in, N))

def generate_gaussian(m, N):
    preact = rng.standard_normal(size=(m, d_in)) @ W
    preact[preact<0] = 0
    return preact

data_fns = (
    lambda m, N : scipy.linalg.hadamard(N)[rng.choice(N,size=(m,),p=np.ones(shape=(N,))/N),:],
    generate_nonuniform_hadamard,
    generate_gaussian,
)

test_Xs = [data_fn(test_m, N) for data_fn in data_fns]


fs = 20
lw = 3
plt.rcParams.update({'font.size': fs})

In [ ]:
train_errors = np.zeros((len(data_fns), len(lambda_fns), len(ms), num_trials))
test_errors = np.zeros((len(data_fns), len(lambda_fns), len(ms), num_trials))

for (i1, data_fn), (i2, lambda_fn), (i3, m), i4 in tqdm(itertools.product(enumerate(data_fns), enumerate(lambda_fns), enumerate(ms), range(num_trials)), total=num_trials*len(ms)*len(data_fns)*len(lambda_fns)):
    X = data_fn(m, N)
    test_X = test_Xs[i1]

    lmbda = lambda_fn(X)

    v = np.linalg.cholesky(np.linalg.inv(X.T @ X + lmbda * np.identity(N)))
    v /= np.diag(v).reshape(1,-1)

    train_errors[i1,i2,i3,i4] = np.linalg.norm(X @ v)**2 / m
    test_errors[i1,i2,i3,i4] = np.linalg.norm(test_X @ v)**2 / test_m

In [ ]:
train_mean = np.mean(train_errors, axis=3)
train_std = np.std(train_errors, axis=3)

test_mean = np.mean(test_errors, axis=3)
test_std = np.std(test_errors, axis=3)

msq_errs = np.tile(np.array([np.linalg.norm(test_X, ord='fro') / test_m for test_X in test_Xs]), (1,len(ms)))

In [ ]:
ymin = [(test_mean[i,:,:] - test_std[i,:,:]).min() for i in range(len(data_fns))]
ymax = [(test_mean[i,:,:] + test_std[i,:,:]).max() for i in range(len(data_fns))]

In [ ]:
for i1 in range(len(data_fns)):
    for i2 in range(2):
        plt.plot(ms, test_mean[i1,i2,:], label=labels[i2], linewidth=lw)
        plt.fill_between(ms, test_mean[i1,i2,:] - test_std[i1,i2,:], test_mean[i1,i2,:] + test_std[i1,i2,:], alpha=0.2)
        
    plt.legend()
    plt.ylabel('Test error')
    plt.xlabel('$m$')
    plt.ylim([ymin[i1] - 0.05*(ymax[i1]-ymin[i1]), ymax[i1] + 0.05*(ymax[i1]-ymin[i1])])
    plt.savefig(f'plots/lambda_comparison_data{i1}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
for i1 in range(len(data_fns)):
    for i2 in range(2,len(labels)):
        plt.plot(ms, test_mean[i1,i2,:], label=labels[i2], linewidth=lw)
        plt.fill_between(ms, test_mean[i1,i2,:] - test_std[i1,i2,:], test_mean[i1,i2,:] + test_std[i1,i2,:], alpha=0.2)
        
    plt.legend()
    plt.ylabel('Test error')
    plt.xlabel('$m$')
    plt.ylim([ymin[i1] - 0.05*(ymax[i1]-ymin[i1]), ymax[i1] + 0.05*(ymax[i1]-ymin[i1])])
    plt.savefig(f'plots/lambda_constant_data{i1}.pdf', bbox_inches='tight')
    plt.show()